# Lab 1b: Linear Regression from Scratch

## Dataset: Student Performance

In this notebook you will build **Linear Regression** completely from scratch (no
`scikit-learn`) using only `numpy`, `pandas` and `matplotlib`.

**Goal:** predict a student's `Final_Score` from `Study_Hours`, `Attendance` and
`Practice_Tests`.

You will:
1. Load the data with **pandas**
2. Explore it and check the **data type** of every column
3. **Split** the data into a training set and a test set with pandas
4. **Normalize** the features
5. Implement the **cost function** and **gradient descent**, train the model, and plot the loss

## Notebooks in this lab
| | from scratch | scikit-learn |
|---|---|---|
| **Linear Regression** | **`linear_regression_scratch.ipynb`** (this one) | `linear_regression_sklearn.ipynb` |
| **Logistic Regression** | `logistic_regression_scratch.ipynb` | `logistic_regression_sklearn.ipynb` |

## Outline
- [1 - Packages](#1)
- [2 - Load the Dataset](#2)
- [3 - Explore & Prepare the Data](#3)
  - [3.1 Data types](#3.1)
  - [3.2 Train / Test split with pandas](#3.2)
  - [3.3 Feature normalization](#3.3)
- [4 - Model function](#4)
- [5 - Cost function](#5)
- [6 - Gradient descent](#6)
- [7 - Training](#7)
- [8 - Results](#8)

<a name="1"></a>
## 1 - Packages

- [numpy](https://www.numpy.org) for vectorized math
- [pandas](https://pandas.pydata.org) to load and manipulate the dataset
- [matplotlib](https://matplotlib.org) to plot the loss and the results

In [ ]:
# On Kaggle / Colab: get the lab files. Skip this cell if you already run the notebook
# from inside the repository folder.
!git clone https://github.com/MLs-labs/Lab_1_ML

In [ ]:
import os
import sys

# Works both on Kaggle (after the git clone above) and locally from the repo folder.
REPO_DIR = "/kaggle/working/Lab_1_ML"
if not os.path.isdir(REPO_DIR):
    REPO_DIR = "."
sys.path.append(REPO_DIR)

import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# public_tests.py contains the checks used throughout this notebook
importlib.invalidate_caches()
from public_tests import *

np.random.seed(1)

<a name="2"></a>
## 2 - Load the Dataset

The dataset `studentPerformance.csv` contains, for each student:
- `Study_Hours`: number of hours studied per day
- `Attendance`: attendance percentage
- `Practice_Tests`: number of practice tests taken
- `Final_Score`: final exam score (0-100) &rarr; **target for Linear Regression**
- `Pass_Fail`: 1 if the student passed, 0 otherwise &rarr; **target for Logistic Regression**

We load it with `pandas.read_csv`, which is the standard way to read a CSV file into a
`DataFrame`.

In [ ]:
df = pd.read_csv(os.path.join(REPO_DIR, "studentPerformance.csv"))

# Always look at your data before doing anything else.
print("Shape of the dataset (rows, columns):", df.shape)
df.head()

<a name="3"></a>
## 3 - Explore & Prepare the Data

<a name="3.1"></a>
### 3.1 Data types

Before building a model, you need to know what you are working with. `df.dtypes` tells you
the type pandas inferred for every column (`int64`, `float64`, `object`, ...). This matters
because a model only understands numbers, so any non-numeric column would need to be
encoded first.

In [ ]:
print(df.dtypes)

# `df.info()` gives a more complete summary: dtypes, non-null counts and memory usage.
df.info()

# Check for missing values.
print("\nMissing values per column:\n", df.isnull().sum())

# Quick statistical summary (mean, std, min, max, quartiles) for every numeric column.
df.describe()

All 5 columns are numeric (`int64` or `float64`) and there are no missing values, so the
dataset is already usable as-is - no encoding or imputation is required.

<a name="3.2"></a>
### 3.2 Train / Test split with pandas

We need a **training set** to fit the model and a separate **test set** to check that it
generalizes to data it has never seen. A simple and common way to do this with pandas is:

1. Shuffle the rows of the DataFrame with `df.sample(frac=1, ...)`
2. Cut the shuffled DataFrame at a chosen index (e.g. 80% train / 20% test) using `.iloc`

`frac=1` means "sample 100% of the rows", which shuffles the whole DataFrame.
`random_state` makes the shuffle reproducible.

In [ ]:
train_ratio = 0.8

### START CODE HERE ### (~ 5 lines of code)
# shuffle df with df.sample(frac=1, random_state=1) and reset_index(drop=True)
df_shuffled = None
# index that marks 80% of the way through df_shuffled
split_index = None
# slice df_shuffled with .iloc to get the train and test DataFrames
df_train = None
df_test = None
### END CODE HERE ###

print(f"Training examples: {len(df_train)}")
print(f"Test examples:     {len(df_test)}")

# Now split each set into input features `X` and targets `y`, and convert them to numpy
# arrays since our from-scratch functions work with numpy, not DataFrames.
feature_cols = ["Study_Hours", "Attendance", "Practice_Tests"]

X_train = df_train[feature_cols].to_numpy()
X_test = df_test[feature_cols].to_numpy()

y_train_reg = df_train["Final_Score"].to_numpy()
y_test_reg = df_test["Final_Score"].to_numpy()

print("X_train shape:", X_train.shape)
print("y_train_reg shape:", y_train_reg.shape)

Run the cell below to check that your train/test split is correct.

In [ ]:
split_test(df, df_train, df_test, X_train, X_test)

<a name="3.3"></a>
### 3.3 Feature normalization

The three features live on very different scales (hours vs. percentage vs. test count).
Gradient descent converges much faster and more reliably when features are on a similar
scale, so we apply **z-score normalization**:
$$ x_{norm} = \frac{x - \mu}{\sigma} $$
The mean `mu` and standard deviation `sigma` are computed **only on the training set** and
then reused to scale the test set, so that no information from the test set leaks into
training.

In [ ]:
def zscore_normalize(X, mu=None, sigma=None):
    """Normalizes the columns of X to zero mean and unit variance."""
    if mu is None or sigma is None:
        mu = np.mean(X, axis=0)
        sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

X_train_norm, mu, sigma = zscore_normalize(X_train)
X_test_norm, _, _ = zscore_normalize(X_test, mu, sigma)

print("Feature means (train):", mu)
print("Feature stds  (train):", sigma)

<a name="4"></a>
## 4 - Model function

For multiple linear regression with `n` features, the model's prediction for one example
$x^{(i)} \in \mathbb{R}^n$ is:
$$ f_{w,b}(x^{(i)}) = w \cdot x^{(i)} + b = \sum_{j=0}^{n-1} w_j x_j^{(i)} + b $$
where `w` is a vector of weights (one per feature) and `b` is the bias (a scalar).

In [ ]:
def predict_linear(X, w, b):
    """
    Computes the linear regression prediction for every row of X.

    Args:
        X (ndarray (m,n)): m examples, n features
        w (ndarray (n,)):  weights
        b (scalar):        bias

    Returns:
        ndarray (m,): predictions f_wb(x) for every example
    """
    return X @ w + b

<a name="5"></a>
## 5 - Cost function

The cost function measures how far the model's predictions are from the actual targets. For
linear regression we use the **mean squared error** (divided by 2 for a cleaner gradient):
$$ J(w,b) = \frac{1}{2m} \sum_{i=0}^{m-1} \left( f_{w,b}(x^{(i)}) - y^{(i)} \right)^2 $$
The smaller `J(w,b)`, the better `w` and `b` fit the data.

In [ ]:
def compute_cost_linear(X, y, w, b):
    """
    Computes the mean squared error cost for linear regression.

    Args:
        X (ndarray (m,n)): m examples, n features
        y (ndarray (m,)):  actual targets
        w (ndarray (n,)):  weights
        b (scalar):        bias

    Returns:
        total_cost (float): the cost J(w,b)
    """
    m = X.shape[0]

    ### START CODE HERE ### (~ 2 lines of code)
    f_wb = None
    total_cost = None
    ### END CODE HERE ###

    return total_cost

# Sanity check with w = 0, b = 0 -> cost should equal the mean squared value of y / 2
w_init = np.zeros(X_train_norm.shape[1])
b_init = 0.0
print("Cost at w=0, b=0:", compute_cost_linear(X_train_norm, y_train_reg, w_init, b_init))

Run the cell below to check your `compute_cost_linear` implementation.

In [ ]:
compute_cost_linear_test(compute_cost_linear)

<a name="6"></a>
## 6 - Gradient descent

Gradient descent repeatedly nudges `w` and `b` in the direction that reduces the cost:
$$ w_j := w_j - \alpha \frac{\partial J(w,b)}{\partial w_j}, \qquad
   b := b - \alpha \frac{\partial J(w,b)}{\partial b} $$
where the partial derivatives (gradients) are:
$$ \frac{\partial J(w,b)}{\partial w_j} = \frac{1}{m} \sum_{i=0}^{m-1}
      \left( f_{w,b}(x^{(i)}) - y^{(i)} \right) x_j^{(i)} $$
$$ \frac{\partial J(w,b)}{\partial b} = \frac{1}{m} \sum_{i=0}^{m-1}
      \left( f_{w,b}(x^{(i)}) - y^{(i)} \right) $$

In [ ]:
def compute_gradient_linear(X, y, w, b):
    """
    Computes the gradient of the linear regression cost with respect to w and b.

    Args:
        X (ndarray (m,n)): m examples, n features
        y (ndarray (m,)):  actual targets
        w (ndarray (n,)):  weights
        b (scalar):        bias

    Returns:
        dj_dw (ndarray (n,)): gradient of the cost w.r.t. w
        dj_db (scalar):       gradient of the cost w.r.t. b
    """
    m = X.shape[0]

    ### START CODE HERE ### (~ 3 lines of code)
    error = None      # f_wb - y, shape (m,)
    dj_dw = None       # shape (n,)
    dj_db = None
    ### END CODE HERE ###

    return dj_dw, dj_db

def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    """
    Performs batch gradient descent to fit w, b.

    Args:
        X, y :                 training data and targets
        w_in, b_in :           initial values of the parameters
        cost_function :        function to compute the cost
        gradient_function :    function to compute the gradient
        alpha (float):         learning rate
        num_iters (int):       number of iterations

    Returns:
        w, b :        parameters found after running gradient descent
        J_history :   cost at every iteration (for plotting)
    """
    w = w_in.copy()
    b = b_in
    J_history = []

    for i in range(num_iters):
        dj_dw, dj_db = gradient_function(X, y, w, b)

        ### START CODE HERE ### (~ 2 lines of code)
        # simultaneously update w and b using the gradient and the learning rate
        w = None
        b = None
        ### END CODE HERE ###

        J_history.append(cost_function(X, y, w, b))

        if i % max(1, num_iters // 10) == 0:
            print(f"Iteration {i:5}: Cost {J_history[-1]:8.4f}")

    return w, b, J_history

Run the cell below to check your `compute_gradient_linear` implementation and the
parameter-update step inside `gradient_descent`.

In [ ]:
compute_gradient_linear_test(compute_gradient_linear)
gradient_descent_test(gradient_descent, compute_cost_linear, compute_gradient_linear)

<a name="7"></a>
## 7 - Training

We run gradient descent on the **normalized training features**.

In [ ]:
alpha = 0.1
num_iters = 1000

w_init = np.zeros(X_train_norm.shape[1])
b_init = 0.0

### START CODE HERE ### (~ 1 line of code)
# call gradient_descent with X_train_norm, y_train_reg, w_init, b_init,
# compute_cost_linear, compute_gradient_linear, alpha and num_iters
w_final, b_final, J_hist_linear = None, None, None
### END CODE HERE ###

print("\nw,b found by gradient descent:", w_final, b_final)

Run the cell below to check that training on the real dataset converges to the expected
parameters.

In [ ]:
train_linear_test(w_final, b_final, J_hist_linear)

<a name="8"></a>
## 8 - Results

#### Plot the loss
The cost should decrease and flatten out as gradient descent converges.

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(J_hist_linear)
plt.title("Linear Regression - Cost vs. Iteration")
plt.xlabel("Iteration")
plt.ylabel("Cost J(w,b)")
plt.show()

#### Evaluate on the test set

In [ ]:
test_cost = compute_cost_linear(X_test_norm, y_test_reg, w_final, b_final)
print(f"Cost on the test set: {test_cost:.4f}")

y_pred_reg = predict_linear(X_test_norm, w_final, b_final)
rmse = np.sqrt(np.mean((y_pred_reg - y_test_reg) ** 2))
print(f"RMSE on the test set: {rmse:.2f} points")

#### Predicted vs. actual scores

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.6)
lims = [min(y_test_reg.min(), y_pred_reg.min()), max(y_test_reg.max(), y_pred_reg.max())]
plt.plot(lims, lims, c="r", linestyle="--", label="perfect prediction")
plt.title("Linear Regression - Predicted vs. Actual Final Score")
plt.xlabel("Actual Final Score")
plt.ylabel("Predicted Final Score")
plt.legend()
plt.show()

**Congratulations!** You built Linear Regression completely from scratch: loading and
splitting data with pandas, implementing the cost function and gradient descent with numpy,
training the model and plotting its loss curve.

Next: `logistic_regression_scratch.ipynb` applies the same machinery to a classification
problem, and `linear_regression_sklearn.ipynb` shows the scikit-learn shortcut.